In [48]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

In [49]:
df = pd.read_csv('data/expense_data_raw.csv')

In [50]:
df.head(10)

,expense_id,employee_id,employee_name,year,month,month_name,quarter,department,expense_category,headcount,budget_amount,actual_amount,variance_amount,variance_pct,variance_status,approver,fiscal_period
0,EXP-000001,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Cloud Infrastructure,180,3215.98,4201.68,985.70,30.65,Overspend,Manager,2023-01
1,EXP-000002,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Software Licenses,180,843.21,972.05,128.84,15.28,Overspend,Manager,2023-01
2,EXP-000003,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Training,180,263.17,338.44,75.27,28.60,Overspend,Manager,2023-01
3,EXP-000004,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Hardware,180,1965.65,2322.02,356.37,18.13,Overspend,Manager,2023-01
4,EXP-000005,EMP-0001,Allison Hill,2023,1,January,Q1,ENGINEERING,Salaries,180,8794.12,10225.80,1431.68,16.28,Overspend,Director,2023-01
5,EXP-000006,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,SaaS Subscriptions,180,289.36,346.39,57.03,19.71,Overspend,Manager,2023-01
6,EXP-000007,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Cloud Infrastructure,180,3405.42,4087.87,682.45,20.04,Overspend,Manager,2023-01
7,EXP-000008,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Hardware,180,652.23,773.41,121.18,18.58,Overspend,Manager,2023-01
8,EXP-000009,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Software Licenses,180,673.81,842.13,168.32,24.98,Overspend,Manager,2023-01
9,EXP-000010,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Training,180,774.97,938.95,163.98,21.16,Overspend,Manager,2023-01


In [51]:
df.isnull().sum()

expense_id             0
employee_id            0
employee_name          0
year                   0
month                  0
month_name             0
quarter                0
department             0
expense_category       0
headcount              0
budget_amount          0
actual_amount       1136
variance_amount        0
variance_pct           0
variance_status        0
approver               0
fiscal_period          0
dtype: int64

### Removing null values in Actual Coloumn by grouped median

In [52]:
df['actual_amount'] = df.groupby(
    ['department','expense_category']
)['actual_amount'].transform(
    lambda x: x.fillna(x.median())
)

In [53]:
df.to_csv("expense_data_cleaned.csv", index=False)

In [54]:
df.isnull().sum()

expense_id          0
employee_id         0
employee_name       0
year                0
month               0
month_name          0
quarter             0
department          0
expense_category    0
headcount           0
budget_amount       0
actual_amount       0
variance_amount     0
variance_pct        0
variance_status     0
approver            0
fiscal_period       0
dtype: int64

In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55388 entries, 0 to 55387
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   expense_id        55388 non-null  object 
 1   employee_id       55388 non-null  object 
 2   employee_name     55388 non-null  object 
 3   year              55388 non-null  int64  
 4   month             55388 non-null  int64  
 5   month_name        55388 non-null  object 
 6   quarter           55388 non-null  object 
 7   department        55388 non-null  object 
 8   expense_category  55388 non-null  object 
 9   headcount         55388 non-null  int64  
 10  budget_amount     55388 non-null  float64
 11  actual_amount     55388 non-null  float64
 12  variance_amount   55388 non-null  float64
 13  variance_pct      55388 non-null  float64
 14  variance_status   55388 non-null  object 
 15  approver          55388 non-null  object 
 16  fiscal_period     55388 non-null  object

In [56]:
df.duplicated().sum()

np.int64(25)

### 25 fully duplicated transaction records were identified and removed to prevent inflation of expense totals and variance metrics

In [57]:
df = df.drop_duplicates()

In [58]:
df.duplicated().sum()

np.int64(0)

In [59]:
df['expense_id'].duplicated().sum()

np.int64(5)

In [60]:
duplicate_ids = df[
    df['expense_id'].duplicated(keep = False)
]
duplicate_ids = duplicate_ids.sort_values('expense_id')
duplicate_ids

,expense_id,employee_id,employee_name,year,month,month_name,quarter,department,expense_category,headcount,budget_amount,actual_amount,variance_amount,variance_pct,variance_status,approver,fiscal_period
273,EXP-000274,EMP-0054,Mark Perez,2023,1,January,Q1,ENGINEERING,SaaS Subscriptions,180,419.60,1015.300,101.00,24.07,Overspend,Manager,2023-01
55361,EXP-000274,EMP-0054,Mark Perez,2023,1,January,Q1,Engineering,SaaS Subscriptions,180,419.60,993.470,101.00,24.07,Overspend,Manager,2023-01
6890,EXP-006891,EMP-0491,Heather Williams,2023,3,March,Q1,LEGAL,Travel,30,2678.90,3628.620,581.86,21.72,Overspend,Manager,2023-03
55360,EXP-006891,EMP-0491,Heather Williams,2023,3,March,Q1,Legal,Travel,30,2678.90,3124.340,581.86,21.72,Overspend,Manager,2023-03
10803,EXP-010804,EMP-0305,Maria Parker,2023,5,May,Q2,MARKETING,Travel,50,4782.43,2655.795,1772.85,37.07,Overspend,Director,2023-05
55373,EXP-010804,EMP-0305,Maria Parker,2023,5,May,Q2,Marketing,Travel,50,4782.43,2749.590,1772.85,37.07,Overspend,Director,2023-05
12472,EXP-012473,EMP-0185,Lindsay Martinez,2023,6,June,Q2,SALES,Travel,80,2012.40,3173.025,329.03,16.35,Overspend,Manager,2023-06
55379,EXP-012473,EMP-0185,Lindsay Martinez,2023,6,June,Q2,Sales,Travel,80,2012.40,3418.620,329.03,16.35,Overspend,Manager,2023-06
29469,EXP-029470,EMP-0367,Courtney Rodriguez,2024,1,January,Q1,Operations,Travel,60,2903.09,2824.670,-114.09,-3.93,On-Track,Manager,2024-01
55375,EXP-029470,EMP-0367,Courtney Rodriguez,2024,1,January,Q1,OPERATIONS,Travel,60,2903.09,3488.040,-114.09,-3.93,On-Track,Manager,2024-01


In [61]:
df['department'] = df['department'].str.title()

In [62]:
df['department'].unique()

array(['Engineering', 'Sales', 'Marketing', 'Operations', 'Product', 'Hr',
       'Finance', 'Legal'], dtype=object)

### Duplicate transaction IDs were investigated manually. Since duplicate IDs contained modified financial values rather than exact copies, I treated them as amended ERP records and retained the latest occurrence while removing earlier versions

In [63]:
df['expense_id'].duplicated().sum()

np.int64(5)

In [64]:
df = df.drop_duplicates(
    subset='expense_id',
    keep='last'
)

In [65]:
df['expense_id'].duplicated().sum()

np.int64(0)

In [66]:
df['calculated variance'] = (
    df['actual_amount'] - df['budget_amount']
)

df[['calculated variance','variance_amount']].tail(10)

,calculated variance,variance_amount
55353,440.01,440.01
55354,2315.33,422.93
55355,887.25,887.25
55356,162.89,162.89
55357,455.18,455.18
55360,445.44,581.86
55361,573.87,101.00
55373,-2032.84,1772.85
55375,584.95,-114.09
55379,1406.22,329.03


In [67]:
df['variance_amount'] = (df['actual_amount'] - df['budget_amount']).round(2)

In [68]:
df['variance_pct'] = (
    (df['variance_amount']/df['budget_amount']) * 100
).round(2)

In [69]:
df['variance_status'] = df['variance_pct'].apply(
    lambda x: 'Overspend' if x > 5 else ('Underspend' if x < -5 else 'On-Track')
)

In [70]:
df['calculated variance'] = (df['actual_amount'] - df['budget_amount']).round(2)
mismatches = df[df['calculated variance'] != df['variance_amount']]
print(f"Mismatches remaining: {len(mismatches)}")

Mismatches remaining: 0


In [71]:
df.drop(columns=['calculated variance'], inplace=True)

In [72]:
df['fiscal_period'] = pd.to_datetime(
    df['fiscal_period'], format='%Y-%m'
)

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55358 entries, 0 to 55379
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   expense_id        55358 non-null  object        
 1   employee_id       55358 non-null  object        
 2   employee_name     55358 non-null  object        
 3   year              55358 non-null  int64         
 4   month             55358 non-null  int64         
 5   month_name        55358 non-null  object        
 6   quarter           55358 non-null  object        
 7   department        55358 non-null  object        
 8   expense_category  55358 non-null  object        
 9   headcount         55358 non-null  int64         
 10  budget_amount     55358 non-null  float64       
 11  actual_amount     55358 non-null  float64       
 12  variance_amount   55358 non-null  float64       
 13  variance_pct      55358 non-null  float64       
 14  variance_status   55358 non

In [74]:
df.head(10)

,expense_id,employee_id,employee_name,year,month,month_name,quarter,department,expense_category,headcount,budget_amount,actual_amount,variance_amount,variance_pct,variance_status,approver,fiscal_period
0,EXP-000001,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Cloud Infrastructure,180,3215.98,4201.68,985.70,30.65,Overspend,Manager,2023-01-01
1,EXP-000002,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Software Licenses,180,843.21,972.05,128.84,15.28,Overspend,Manager,2023-01-01
2,EXP-000003,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Training,180,263.17,338.44,75.27,28.60,Overspend,Manager,2023-01-01
3,EXP-000004,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Hardware,180,1965.65,2322.02,356.37,18.13,Overspend,Manager,2023-01-01
4,EXP-000005,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,Salaries,180,8794.12,10225.80,1431.68,16.28,Overspend,Director,2023-01-01
5,EXP-000006,EMP-0001,Allison Hill,2023,1,January,Q1,Engineering,SaaS Subscriptions,180,289.36,346.39,57.03,19.71,Overspend,Manager,2023-01-01
6,EXP-000007,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Cloud Infrastructure,180,3405.42,4087.87,682.45,20.04,Overspend,Manager,2023-01-01
7,EXP-000008,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Hardware,180,652.23,773.41,121.18,18.58,Overspend,Manager,2023-01-01
8,EXP-000009,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Software Licenses,180,673.81,842.13,168.32,24.98,Overspend,Manager,2023-01-01
9,EXP-000010,EMP-0002,Noah Rhodes,2023,1,January,Q1,Engineering,Training,180,774.97,938.95,163.98,21.16,Overspend,Manager,2023-01-01


In [75]:
df['actual_amount'].describe()

count    55358.000000
mean      2886.860860
std       3472.601987
min          0.000000
25%        774.772500
50%       1536.805000
75%       3506.475000
max      24003.220000
Name: actual_amount, dtype: float64

In [76]:
(df['actual_amount'] == 0).sum()

np.int64(10)

In [77]:
zero_rows = df[df['actual_amount'] == 0]
print(zero_rows[['expense_id','department','expense_category','budget_amount','actual_amount','variance_status']])

       expense_id   department      expense_category  budget_amount  \
1221   EXP-001222        Sales      Commission Tools         564.47   
5549   EXP-005550        Sales              Training       -1337.91   
10853  EXP-010854   Operations              Software         227.85   
20909  EXP-020910  Engineering  Cloud Infrastructure        4834.28   
23058  EXP-023059  Engineering     Software Licenses        -474.51   
28501  EXP-028502  Engineering    SaaS Subscriptions        -556.33   
36008  EXP-036009    Marketing    Content Production       -3692.57   
37884  EXP-037885        Sales              Training        1500.69   
49305  EXP-049306  Engineering              Training        1182.85   
52401  EXP-052402   Operations       Office Supplies        -188.85   

       actual_amount variance_status  
1221             0.0      Underspend  
5549             0.0      Underspend  
10853            0.0      Underspend  
20909            0.0      Underspend  
23058            0.0   

In [78]:
df = df[df['actual_amount'] != 0]
print(f"Zero rows remaining: {(df['actual_amount'] == 0).sum()}")
print(f"Rows after removing zeros: {len(df)}")

Zero rows remaining: 0
Rows after removing zeros: 55348


### Removed 10 rows where actual_amount = 0.These represent cancelled expenses not removed from the system.Zero spend records are invalid for variance analysis a zero actual against a non-zero budget creates artificial,100% underspend which distorts department variance calculations.

In [79]:
Q1 = df['actual_amount'].quantile(0.25)
Q3 = df['actual_amount'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

print('Q1:',Q1)
print('Q3:',Q3)
print('IQR:',IQR)
print('Lower Bound:',lower_bound)
print('Upper Bound:',upper_bound)

Q1: 775.015
Q3: 3507.215
IQR: 2732.2000000000003
Lower Bound: -3323.2850000000003
Upper Bound: 7605.515


In [80]:
outliers = df[
    df['actual_amount'] > upper_bound
]

print('Total Outliers:',len(outliers))

Total Outliers: 5438


In [81]:
df['is_outlier'] = df['actual_amount'] > upper_bound

outlier_by_dept = df[df['is_outlier'] == True].groupby('department').size()
print(outlier_by_dept)

department
Engineering    4354
Finance         102
Hr               36
Legal           429
Marketing       495
Operations       16
Sales             6
dtype: int64


In [82]:
# Step 1 — Count outliers per department
outlier_counts = (
    df[df['is_outlier'] == True]
    .groupby('department')
    .size()
    .reset_index(name='outlier_count')
)

# Step 2 — Count total rows per department separately
total_counts = (
    df.groupby('department')
    .size()
    .reset_index(name='total_rows')
)

# Step 3 — Merge on department — aligns correctly
outlier_pct_by_dept = total_counts.merge(
    outlier_counts, 
    on='department', 
    how='left'
)

# Step 4 — Fill departments with zero outliers
outlier_pct_by_dept['outlier_count'] = (
    outlier_pct_by_dept['outlier_count'].fillna(0).astype(int)
)

# Step 5 — Calculate percentage
outlier_pct_by_dept['outlier_pct'] = (
    outlier_pct_by_dept['outlier_count'] / 
    outlier_pct_by_dept['total_rows'] * 100
).round(2)

print(outlier_pct_by_dept.sort_values('outlier_pct', ascending=False))

    department  total_rows  outlier_count  outlier_pct
0  Engineering       21619           4354        20.14
3        Legal        2892            429        14.83
4    Marketing        5990            495         8.26
1      Finance        2157            102         4.73
2           Hr        2132             36         1.69
5   Operations        5792             16         0.28
7        Sales       10882              6         0.06
6      Product        3884              0         0.00


In [83]:
# Recalculate all derived columns one final time
# after zero removal and outlier flagging

df['variance_amount'] = (
    df['actual_amount'] - df['budget_amount']
).round(2)

df['variance_pct'] = (
    (df['variance_amount'] / df['budget_amount']) * 100
).round(2)

df['variance_status'] = df['variance_pct'].apply(
    lambda x: 'Overspend' if x > 5 
    else ('Underspend' if x < -5 
    else 'On-Track')
)

print("Derived columns recalculated successfully")
print(df[['budget_amount','actual_amount',
          'variance_amount','variance_pct',
          'variance_status']].head(5))

Derived columns recalculated successfully
   budget_amount  actual_amount  variance_amount  variance_pct variance_status
0        3215.98        4201.68           985.70         30.65       Overspend
1         843.21         972.05           128.84         15.28       Overspend
2         263.17         338.44            75.27         28.60       Overspend
3        1965.65        2322.02           356.37         18.13       Overspend
4        8794.12       10225.80          1431.68         16.28       Overspend


In [84]:
print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)
print(f"Final row count:           {len(df)}")
print(f"Null values remaining:     {df.isnull().sum().sum()}")
print(f"Duplicate rows remaining:  {df.duplicated().sum()}")
print(f"Zero actual amounts:       {(df['actual_amount'] == 0).sum()}")
print(f"Negative budgets:          {(df['budget_amount'] < 0).sum()}")
print(f"Outliers flagged:          {df['is_outlier'].sum()}")
print(f"Columns:                   {list(df.columns)}")
print("=" * 50)

CLEANING SUMMARY
Final row count:           55348
Null values remaining:     0
Duplicate rows remaining:  0
Zero actual amounts:       0
Negative budgets:          0
Outliers flagged:          5438
Columns:                   ['expense_id', 'employee_id', 'employee_name', 'year', 'month', 'month_name', 'quarter', 'department', 'expense_category', 'headcount', 'budget_amount', 'actual_amount', 'variance_amount', 'variance_pct', 'variance_status', 'approver', 'fiscal_period', 'is_outlier']


In [85]:
df['expense_band'] = pd.cut(
    df['actual_amount'],
    bins=[0, 500, 2000, 10000, float('inf')],
    labels=[
        'Low Spend',      # $0     - $500    office supplies, small SaaS
        'Medium Spend',   # $500   - $2,000  training, small travel
        'High Spend',     # $2,000 - $10,000 events, larger travel, software
        'Critical Spend'  # $10,000+         cloud infra, legal fees, audits
    ]
)

print(df['expense_band'].value_counts())

expense_band
Medium Spend      25578
High Spend        18774
Low Spend          7906
Critical Spend     3090
Name: count, dtype: int64


In [86]:
df['expense_band'].value_counts()

expense_band
Medium Spend      25578
High Spend        18774
Low Spend          7906
Critical Spend     3090
Name: count, dtype: int64

In [87]:
def variance_severity(x):
    if x > 25:
        return 'Critical Overspend'
    elif x > 10:
        return 'Moderate Overspend'
    elif x > 5:
        return 'Mild Overspend'
    elif x < -15:
        return 'High Underspend'
    elif x < -5:
        return 'Moderate Underspend'
    else:
        return 'Normal'
    
df['variance_severity'] = df['variance_pct'].apply(variance_severity)

print(df['variance_severity'].value_counts())

variance_severity
Moderate Overspend     24092
Critical Overspend     15037
Normal                 12424
Moderate Underspend     2465
Mild Overspend          1047
High Underspend          283
Name: count, dtype: int64


In [88]:
df['budget_utilization_pct'] = (
    df['actual_amount'] /
    df['budget_amount']
) * 100

df['budget_utilization_pct'] = (
    df['budget_utilization_pct']
    .round(2)
)

In [89]:
df['budget_utilization_pct'].describe()

count    55348.000000
mean       117.020014
std         19.201322
min         40.330000
25%        103.470000
50%        119.880000
75%        125.490000
max        901.050000
Name: budget_utilization_pct, dtype: float64

In [90]:
def utilization_category(x):
    if x > 150:
        return 'Critical Over Utilized'
    elif x > 125:
        return 'High Over Utilized'
    elif x > 110:
        return 'Over Utilized'
    elif x >= 90:
        return 'Efficient'
    elif x >= 75:
        return 'Under Utilized'
    else:
        return 'Critically Under Utilized'
    
df['utilization_category'] = (
    df['budget_utilization_pct']
    .apply(utilization_category)
)

print(df['utilization_category'].value_counts())
print(df['utilization_category'].value_counts(normalize=True).round(2))

utilization_category
Over Utilized                24092
Efficient                    15845
High Over Utilized           14648
Critical Over Utilized         389
Critically Under Utilized      189
Under Utilized                 185
Name: count, dtype: int64
utilization_category
Over Utilized                0.44
Efficient                    0.29
High Over Utilized           0.26
Critical Over Utilized       0.01
Critically Under Utilized    0.00
Under Utilized               0.00
Name: proportion, dtype: float64


In [91]:
df.to_csv(
    'expense_data_cleaned_final.csv',
    index=False
)

print("Final dataset exported successfully")

Final dataset exported successfully
